All left-going plays are transformed so that offense always moves to the right. For left-going plays, we apply a 180° rotation to the coordinate system: x = 120 - x, y = 160/3 - y, and the angle features are rotated by 180°, so dir = (dir + 180) % 360 and o = (o + 180) % 360. We apply the same rule to landing coordinates: ball_land_x = 120 - ball_land_x and ball_land_y = 160/3 - ball_land_y. We also transform absolute_yardline_number as 120 - absolute_yardline_number. After this, all plays are stored with play_direction = right.

In [ ]:
import pandas as pd
import numpy as np

FIELD_LENGTH = 120.0
FIELD_WIDTH = 160.0 / 3.0  # 53.3333...


def _rotate_180_for_left(values, is_left, field_limit):
    values = values.astype(float)
    return np.where(is_left, field_limit - values, values)


def _rotate_angle_180_for_left(values, is_left):
    values = values.astype(float)
    return np.where(is_left, (values + 180.0) % 360.0, values)


def standardize_offense_to_right(
    df: pd.DataFrame,
    play_direction_col: str = "play_direction",
    copy: bool = True,
    preserve_original_direction: bool = True,
    add_was_flipped: bool = True,
) -> pd.DataFrame:
    if play_direction_col not in df.columns:
        raise ValueError(
            f"Column '{play_direction_col}' not found. "
            "For tables without play_direction, first merge in a play-level "
            "direction map and then run this function."
        )

    out = df.copy() if copy else df

    direction = out[play_direction_col].astype(str).str.lower().str.strip()
    is_left = direction.eq("left")

    if preserve_original_direction and "original_play_direction" not in out.columns:
        out["original_play_direction"] = out[play_direction_col]

    if add_was_flipped:
        out["was_flipped"] = is_left

    if "x" in out.columns:
        out["x"] = _rotate_180_for_left(out["x"], is_left, FIELD_LENGTH)

    if "y" in out.columns:
        out["y"] = _rotate_180_for_left(out["y"], is_left, FIELD_WIDTH)

    if "ball_land_x" in out.columns:
        out["ball_land_x"] = _rotate_180_for_left(out["ball_land_x"], is_left, FIELD_LENGTH)

    if "ball_land_y" in out.columns:
        out["ball_land_y"] = _rotate_180_for_left(out["ball_land_y"], is_left, FIELD_WIDTH)

    if "absolute_yardline_number" in out.columns:
        out["absolute_yardline_number"] = _rotate_180_for_left(
            out["absolute_yardline_number"], is_left, FIELD_LENGTH
        )

    for angle_col in ["dir", "o"]:
        if angle_col in out.columns:
            out[angle_col] = _rotate_angle_180_for_left(out[angle_col], is_left)

    out[play_direction_col] = "right"
    return out


def build_play_direction_map(
    input_df: pd.DataFrame,
    game_col: str = "game_id",
    play_col: str = "play_id",
    play_direction_col: str = "play_direction",
) -> pd.DataFrame:
    needed = [game_col, play_col, play_direction_col]
    missing = [c for c in needed if c not in input_df.columns]
    if missing:
        raise ValueError(f"Missing columns in input_df: {missing}")

    play_map = input_df[[game_col, play_col, play_direction_col]].drop_duplicates()

    dup_check = play_map.groupby([game_col, play_col])[play_direction_col].nunique()
    bad = dup_check[dup_check > 1]
    if len(bad) > 0:
        raise ValueError(
            "Found plays with multiple play_direction values. "
            "Expected exactly one direction per play."
        )

    return play_map


def standardize_output_with_input_reference(
    output_df: pd.DataFrame,
    input_df: pd.DataFrame,
    game_col: str = "game_id",
    play_col: str = "play_id",
    play_direction_col: str = "play_direction",
    copy: bool = True,
) -> pd.DataFrame:
    play_map = build_play_direction_map(
        input_df=input_df,
        game_col=game_col,
        play_col=play_col,
        play_direction_col=play_direction_col,
    )

    out = output_df.copy() if copy else output_df
    out = out.merge(play_map, on=[game_col, play_col], how="left", validate="many_to_one")

    if out[play_direction_col].isna().any():
        missing_pairs = out.loc[out[play_direction_col].isna(), [game_col, play_col]].drop_duplicates()
        raise ValueError(
            "Some output plays could not find play_direction in input_df. "
            f"Missing play pairs example:\n{missing_pairs.head()}"
        )

    out = standardize_offense_to_right(
        out,
        play_direction_col=play_direction_col,
        copy=False,
        preserve_original_direction=True,
        add_was_flipped=True,
    )

    return out

In [ ]:
import pandas as pd

base_path = "/content/drive/MyDrive/Data Science"

input_df = pd.read_csv(f"{base_path}/input_2023_w01.csv")
input_std = standardize_offense_to_right(input_df)
input_std.to_csv(f"{base_path}/input_2023_w01_standardized.csv", index=False)

output_df = pd.read_csv(f"{base_path}/output_2023_w01.csv")
output_std = standardize_output_with_input_reference(output_df, input_df)
output_std.to_csv(f"{base_path}/output_2023_w01_standardized.csv", index=False)

print("done")

done
